In [0]:
SELECT *
FROM com_edp_prd.com_raw.vod_references
WHERE reference_type = 'Specialty'
AND (
    name ILIKE '%Clinical Pharmacology%' OR
    name ILIKE '%Pharmacology%' OR
    name ILIKE '%Pharmacy Specialty%' OR
    name ILIKE '%Pharmaceutical Medicine%'
);

In [0]:
/* ================= GET HCP VID ================= */
WITH hcp_vid AS (
    SELECT DISTINCT
        vid__v AS hcp_vid,
        npi_num__v AS hcp_npi,
        first_name_cda__v,
        last_name_cda__v
    FROM com_edp_prd.com_raw.vod_hcp
    WHERE first_name_cda__v ILIKE '%neil%'
      AND last_name_cda__v ILIKE '%patel%'
),

/* ================= VOD AFFILIATION ================= */
vod_ranked AS (
    SELECT
        a.hcp_vid,
        a.hcp_npi,
        c.npi_num__v AS hco_npi,
        c.corporate_name__v AS hco_name,
        ROW_NUMBER() OVER (
            PARTITION BY a.hcp_vid
            ORDER BY 
                b.modified_date__v DESC NULLS LAST,
                b.status_update_time__v DESC NULLS LAST
        ) AS rn
    FROM hcp_vid a
    LEFT JOIN com_edp_prd.com_raw.vod_parenthco b
        ON a.hcp_vid = b.entity_vid__v
       AND b.hierarchy_type__v = 'HCP_HCO'
    LEFT JOIN com_edp_prd.com_raw.vod_hco c
        ON b.parent_hco_vid__v = c.vid__v
    WHERE b.parent_hco_status__v = 'A'
      AND b.relationship_type__v = '7356'
),

vod_final AS (
    SELECT
        hcp_vid,
        hcp_npi,
        hco_npi,
        hco_name
    FROM vod_ranked
    WHERE rn = 1
)

SELECT *
FROM vod_final;

In [0]:
('243053469240394756', '941662202607438687')

In [0]:
CREATE OR REPLACE TEMPORARY view pharmacist_hcps AS 

WITH specialty_ref AS (
    SELECT code
    FROM com_edp_prd.com_raw.vod_references
    WHERE reference_type = 'Specialty'
      AND (
            name ILIKE '%Clinical Pharmacology%' OR
            name ILIKE '%Pharmacology%' OR
            name ILIKE '%Pharmacy Specialty%' OR
            name ILIKE '%Pharmaceutical Medicine%'
      )
),

/* ================= HCP BASE ================= */
hcp_base AS (
    SELECT DISTINCT
        h.npi_num__v AS hcp_npi,
        h.first_name__v,
        h.last_name__v,
        h.specialty_1__v
    FROM com_edp_prd.com_raw.vod_hcp h
    INNER JOIN specialty_ref s
        ON h.specialty_1__v = s.code
    WHERE h.npi_num__v IS NOT NULL
),

/* ================= GET VID ================= */
hcp_vid AS (
    SELECT 
        a.*,
        b.vid__v AS hcp_vid
    FROM hcp_base a
    LEFT JOIN com_edp_prd.com_raw.vod_hcp b
        ON a.hcp_npi = b.npi_num__v
),

/* ================= VOD AFFILIATION ================= */
vod_ranked AS (
    SELECT
        a.hcp_npi,
        c.npi_num__v AS vod_hco_npi,
        c.corporate_name__v AS vod_hco_name,
        ROW_NUMBER() OVER (
            PARTITION BY a.hcp_npi
            ORDER BY 
                b.modified_date__v DESC NULLS LAST,
                b.status_update_time__v DESC NULLS LAST
        ) AS rn
    FROM hcp_vid a
    LEFT JOIN com_edp_prd.com_raw.vod_parenthco b
        ON a.hcp_vid = b.entity_vid__v
       AND b.hierarchy_type__v = 'HCP_HCO'
    LEFT JOIN com_edp_prd.com_raw.vod_hco c
        ON b.parent_hco_vid__v = c.vid__v
    WHERE b.parent_hco_status__v = 'A'
      AND b.relationship_type__v = '7356'
),

vod_final AS (
    SELECT
        hcp_npi,
        vod_hco_npi,
        vod_hco_name
    FROM vod_ranked
    WHERE rn = 1
),

/* ================= KOMODO FALLBACK ================= */
komodo AS (
    SELECT 
        a.hcp_npi,
        b.hco_primary_npi AS komodo_hco_npi,
        c.organization_name AS komodo_hco_name
    FROM hcp_base a
    LEFT JOIN com_edp_prd.com_raw.kom_providers b
        ON a.hcp_npi = b.npi
       AND b.provider_type = 'INDIVIDUAL'
    LEFT JOIN com_edp_prd.com_raw.kom_providers c
        ON b.hco_primary_npi = c.npi
       AND c.provider_type = 'ORGANIZATION'
),

/* ================= FINAL BASE ================= */
base_output AS (
    SELECT 
        a.hcp_npi,
        a.first_name__v,
        a.last_name__v,
        a.specialty_1__v AS hcp_specialty,

        COALESCE(v.vod_hco_npi, k.komodo_hco_npi, '-') AS hco_npi,
        COALESCE(v.vod_hco_name, k.komodo_hco_name, '-') AS hco_name,

        CASE 
            WHEN v.vod_hco_npi IS NOT NULL THEN 'VOD'
            WHEN k.komodo_hco_npi IS NOT NULL THEN 'Komodo'
            ELSE 'None'
        END AS affiliation_source

    FROM hcp_base a
    LEFT JOIN vod_final v ON a.hcp_npi = v.hcp_npi
    LEFT JOIN komodo k ON a.hcp_npi = k.hcp_npi
),

/* ================= VOD HCO ADDRESS ================= */
hco_vod_address AS (
    SELECT
        h.npi_num__v AS hco_npi,
        a.address_line_1__v,
        a.postal_code_cda__v,
        ROW_NUMBER() OVER (
            PARTITION BY h.npi_num__v
            ORDER BY a.modified_date__v DESC
        ) AS rn
    FROM com_edp_prd.com_raw.vod_hco h
    JOIN com_edp_prd.com_raw.vod_address a
        ON a.entity_vid__v = h.vid__v
       AND a.entity_type__v = 'HCO'
       AND a.record_state__v = 'VALID'
       AND a.address_status__v IN ('A','DS')
       AND a.address_verification_status__v NOT IN ('NS','U')
),

/* ================= FINAL ENRICHMENT ================= */
final_output AS (
    SELECT
        b.*,

        /* Address fallback logic */
        CASE 
            WHEN v.postal_code_cda__v IS NOT NULL THEN v.address_line_1__v
            ELSE kp.provider_address
        END AS hco_address,

        COALESCE(v.postal_code_cda__v, kp.provider_zip) AS hco_zip,
        z.city AS hco_city,
        z.state AS hco_state

    FROM base_output b

    LEFT JOIN hco_vod_address v
        ON b.hco_npi = v.hco_npi
       AND v.rn = 1

    LEFT JOIN com_edp_prd.com_raw.kom_providers kp
        ON b.hco_npi = kp.npi
       AND kp.provider_type = 'ORGANIZATION'

    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
        ON COALESCE(v.postal_code_cda__v, kp.provider_zip) = z.zipcode
)

SELECT distinct *
FROM final_output
where hco_name != '-';

In [0]:
CREATE OR REPLACE TEMP VIEW reporting_parent AS
SELECT * FROM VALUES
  ('children''s hospital colorado', '13123 east 16th avenue', '80045', 'tier 1'),
  ('westchester medical center', '100 woods rd valhalla', '10595', 'tier 1'),
  ('kaiser socal', '9455 clairemont mesa blvd', '92123', 'tier 1'),
  ('seattle children''s hospital', '4800 sand point way ne', '98105', 'tier 1'),
  ('university of new mexico hospitals', '2211 lomas blvd.', '87106', 'tier 1'),
  ('childrens hospital of philadelphia', '3401 civic center blvd philadelphia', '19104', 'tier 1'),
  ('cincinnati childrens', '3333 burnet ave cincinnati', '45229', 'tier 1'),
  ('ucsf benioff childrens hospital-oakland', '747 52nd st oakland', '94609', 'tier 1'),
  ('childrens hospital of orange county', '1201 w la veta ave orange', '92868', 'tier 1'),
  ('ut health', '6411 fannin st', '77030', 'tier 1'),
  ('cook children''s medical center', '801 7th ave fort worth', '76104', 'tier 1'),
  ('boston children''s hospital', '300 longwood ave', '02115', 'tier 1'),
  ('east tennessee childrens hospital', '2018 w clinch ave knoxville', '37916', 'tier 1'),
  ('children''s nebraska', '8200 dodge st', '68114', 'tier 1'),
  ('west virginia university hospitals', '1 medical center dr', '26505', 'tier 1'),
  ('nyu langone medical center', '550 1st ave', '10016', 'tier 1'),
  ('texas childrens hospital', '6621 fannin st houston', '77030', 'tier 1'),
  ('usf health (university medical service association)', '13101 bruce b. downs blvd.', '33612', 'tier 1'),
  ('akron children''s hospital', 'one perkins square akron', '44308', 'tier 1'),
  ('riley children''s health', '705 riley hospital dr', '46202', 'tier 1'),
  ('lurie children''s hospital of chicago', '225 e chicago avenue chicago', '60611', 'tier 1'),
  ('children''s wisconsin', '8915 w connell ave milwaukee', '53226', 'tier 1'),
  ('university of iowa healthcare medical center', '200 hawkins dr iowa city', '52242', 'tier 1'),
  ('university of miami hospital and clinics', '1475 nw 12th ave', '33136', 'tier 1'),
  ('johns hopkins hospital', '1800 orleans st', '21287', 'tier 1'),
  ('inova', '3300 gallows rd', '22042', 'tier 1'),
  ('uc davis hospital', '2315 stockton blvd ste 2100 sacramento', '95817', 'tier 1'),
  ('columbia- cornell weill medicine', '622 w 168th st new york', '10032', 'tier 1'),
  ('university of michigan hospitals and health centers', '1500 e medical center dr', '48109', 'tier 1'),
  ('primary childrens hospital-salt lake city', '100 n mario capecchi dr salt lake city', '84113', 'tier 1'),
  ('university of north carolina hospitals', '101 manning dr', '27514', 'tier 1'),
  ('orlando health orlando regional medical center', '52 w underwood st orlando', '32806', 'tier 1'),
  ('uams medical center', '4301 w markham st', '72205', 'tier 1'),
  ('detroit medical center - dmc', '3901 beaubien st', '48201', 'tier 1'),
  ('atrium health', '1000 blythe blvd', '28203', 'tier 1'),
  ('ucla health', '10833 le conte ave los angeles', '90095', 'tier 1'),
  ('kaiser norcal', '901 nevin ave richmond', '94801', 'tier 1'),
  ('valley children''s healthcare', '9300 valley childrens pl', '93636', 'tier 1'),
  ('lucile packard children''s hospital stanford', '725 welch rd', '94304', 'tier 1'),
  ('university of rochester', '601 elmwood avenue', '14642', 'tier 1'),
  ('emory university hospital', '1364 clifton rd ne', '30322', 'tier 1'),
  ('childrens hospital los angeles', '4650 w sunset blvd los angeles', '90027', 'tier 1'),
  ('vanderbilt- monroe carell jr children''s hospital', '2200 childrens way rm 4508 nashville', '37232', 'tier 1'),
  ('children''s health texas', '1935 medical district dr', '75235', 'tier 1'),
  ('childrens of alabama', '1600 7th ave s birmingham', '35233', 'tier 1'),
  ('nationwide children''s hospital', '700 childrens dr', '43205', 'tier 1'),
  ('mount sinai hospital', '1 gustave l levy pl', '10029', 'tier 1'),
  ('st. joseph''s healthcare system', '703 main st', '07503', 'tier 1'),
  ('yale-new haven hospital', '20 york street', '06504', 'tier 1'),
  ('oklahoma children''s hospital ou health', '1200 n. children''s ave', '73104', 'tier 1'),
  ('hackensack university medical center', '30 prospect avenue', '07601', 'tier 1'),
  ('university of wisconsin', '7974 uw health ct middleton', '53562', 'tier 1'),
  ('childrens national hospital', '111 michigan ave nw washington', '20010', 'tier 1'),
  ('christus childrens san antonio', '333 n santa rosa san antonio', '78207', 'tier 1'),
  ('phoenix children''s hospital', '1919 e thomas rd', '85016', 'tier 1'),
  ('saint peter''s university hospital', '254 easton ave', '08901', 'tier 1'),
  ('upmc children''s hospital of pittsburgh', '4401 penn avenue', '15224', 'tier 1'),
  ('university of illinois hospital health sciences system', '1740 w taylor st', '60612', 'tier 1'),
  ('ohsu hospital portland', '3181 sw sam jackson park rd portland', '97239', 'tier 1'),
  ('corewell health system', '100 michigan st. ne', '49503', 'tier 1'),
  ('uf health', '1600 sw archer rd', '32610', 'tier 1'),
  ('multicare mary bridge children''s hospital', '317 martin luther king jr way', '98405', 'tier 1'),
  ('m health fairview', '2450 riverside ave minneapolis', '55454', 'tier 1'),
  ('parkland health hospital system', '5201 harry hines blvd', '75235', 'tier 1'),
  ('musc health', '201 w meeting st ste a lancaster', '29720', 'tier 1'),
  ('childrens mercy hospital', '2401 gillham rd kansas city', '64108', 'tier 1'),
  ('legacy emanuel health system', '2801 n. gantenbein ave.', '97227', 'tier 1'),
  ('nyc health + hospitals metropolitan', '1901 1st ave', '10029', 'tier 1'),
  ('university of virginia medical center', '1215 lee st', '22908', 'tier 1'),
  ('rady childrens hospital san diego', '8010 frost st san diego', '92123', 'tier 1'),
  ('the greenwood genetic center', '106 gregor mendel cir greenwood', '29646', 'tier 1'),
  ('kentucky childrens hospital', '800 rose st lexington', '40536', 'tier 1'),
  ('mayo clinic hospital - rochester', '1216 2nd st sw', '55902', 'tier 1'),
  ('maine medical center', '22 bramhall st', '04102', 'tier 1'),
  ('university of louisville', '530 south jackson street', '40202', 'tier 1'),
  ('upstate university hospital', '750 e adams st', '13210', 'tier 1'),
  ('st. louis children''s hospital', 'one children''s place', '63110', 'tier 1'),
  ('children''s hospital of new orleans', '200 henry clay ave new orleans', '70118', 'tier 1'),
  ('driscoll childrens hospital-corpus christi', '3533 s alameda st corpus christi', '78411', 'tier 1'),
  ('norton children''s hospital', '1930 bishop ln', '40218', 'tier 2'),
  ('alfred i. dupont hospital for children', '1600 rockland rd', '19803', 'tier 2'),
  ('shodair childrens hospital', '2755 colonial dr, helena', '59601', 'tier 2'),
  ('beaumont hospital - royal oak', '3601 w 13 mile rd', '48073', 'tier 2'),
  ('cleveland clinic', '9500 euclid ave', '44195', 'tier 2'),
  ('sanford usd medical center', '1305 w 18th st', '57105', 'tier 2'),
  ('university of mississippi medical center', '2500 n state st', '39216', 'tier 2'),
  ('nemours children''s hospital', '13535 nemours pkwy', '32827', 'tier 2'),
  ('ohio state university wexner medical center', '370 w 9th ave', '43210', 'tier 2'),
  ('university hospitals cleveland medical center', '11100 euclid ave', '44106', 'tier 2'),
  ('barnes-jewish hospital', '1 barnes jewish hospital plz saint louis', '63110', 'tier 2'),
  ('tulane university school of medicine', '1430 tulane ave new orleans', '70112', 'tier 2'),
  ('st. vincent indianapolis hospital', '2001 w 86th st', '46260', 'tier 2'),
  ('baptist medical center', '111 dallas st', '78205', 'tier 2'),
  ('rush university medical center', '1645 w jackson boulevard suite 215 chicago', '60612', 'tier 2'),
  ('stanford university health', '725 welch rd palo alto', '94304', 'tier 2'),
  ('university of california at irvine health-orange', '101 the city dr s orange', '92868', 'tier 2'),
  ('university of colorado hospital', '12605 e 16th ave', '80045', 'tier 2'),
  ('palmetto health richland', '5 richland medical park dr', '29203', 'tier 2'),
  ('st luke''s medical center wood river ltd', '100 hospital drive, hailey', '83340', 'tier 2'),
  ('baptist hospital of miami', '8900 n kendall dr', '33176', 'tier 2'),
  ('ssm health saint louis university hospital', '3635 vista ave', '63110', 'tier 2'),
  ('washington university', '660 s euclid ave. saint louis', '63110', 'tier 2'),
  ('slidell memorial hospital', '1001 gause blvd', '70458', 'tier 2'),
  ('modesto city hospital', '1700 coffee rd', '95355', 'tier 2'),
  ('childrens minnesota hospital-minneapolis', '2525 chicago avenue south, minneapolis', '55404', 'tier 2'),
  ('penn state milton s. hershey medical center', '500 university dr', '17033', 'tier 2'),
  ('the medical college of wisconsin inc.', '8701 watertown plank rd', '53226', 'tier 2'),
  ('children''s healthcare of atlanta at scottish rite', '1001 johnson ferry rd ne', '30342', 'tier 2'),
  ('cure for the kids', '1 breakthrough way, las vegas, nv 89135', '89135', 'tier 2'),
  ('university of kentucky', '800 rose st', '40536', 'tier 2'),
  ('texas scottish rite hospital for children', '2222 welborn st', '75219', 'tier 2'),
  ('nicklaus children''s hospital', '3100 sw 62nd ave', '33155', 'tier 2'),
  ('duke university hospital', '2301 erwin rd', '27705', 'tier 2'),
  ('gillette children''s specialty healthcare', '200 university ave e', '55101', 'tier 2'),
  ('reading hospital', 's 6th ave at spruce st', '19611', 'tier 2'),
  ('northwell health', '300 community dr', '11030', 'tier 2'),
  ('dell children''s medical center of central texas', '4900 mueller blvd', '78723', 'tier 2'),
  ('carilion medical group', '282 westlake rd', '24101', 'tier 2'),
  ('baylor college of medicine-department of pediatrics', '1 baylor plz houston', '77030', 'tier 2'),
  ('northwestern medicine', '10400 haligus road huntley', '60142', 'tier 2'),
  ('erlanger medical center', '975 e 3rd st', '37403', 'tier 2'),
  ('montefiore medical center', '111 e 210th st', '10467', 'tier 2'),
  ('ach physician services', '1 childrens way', '72202', 'tier 2'),
  ('albany medical center', '43 new scotland ave', '12208', 'tier 2'),
  ('women and infants hospital of rhode island', '101 dudley st', '02905', 'tier 2'),
  ('sentara norfolk general hospital', '600 gresham dr', '23507', 'tier 2'),
  ('sequence md', '1601 e 19th ave ste 6450, denver', '80218', 'tier 2'),
  ('ut medical group inc.', '1407 union ave', '38104', 'tier 2'),
  ('cedars-sinai medical center', '8700 beverly blvd', '90048', 'tier 2'),
  ('stony brook hospital', 'stony brook hospital medical, nicholls road', '11794', 'tier 2'),
  ('providence sacred heart medical center', '101 w 8th ave', '99204', 'tier 2'),
  ('maricopa integrated health system', '2601 e roosevelt st', '85008', 'tier 2'),
  ('queen''s medical center', '1301 punchbowl st', '96813', 'tier 2'),
  ('unitypoint health - iowa methodist medical center', '1200 pleasant st', '50309', 'tier 2'),
  ('saint joseph hospital', '1835 franklin st', '80218', 'tier 2'),
  ('jersey shore university medical center', '1945 state route 33', '07753', 'tier 2'),
  ('boys town national research hospital downtown clinic', '555 n 30th st, omaha', '68131', 'tier 2'),
  ('lee health', '13681 doctors way, fort myers', '33912', 'tier 2'),
  ('university of kentucky', '1000 s limestone', '40536', 'tier 2'),
  ('memorial hospital east', '1404 cross st', '62269', 'tier 2'),
  ('ucsd health', '200 w arbor dr rm 1317', '92103', 'tier 2'),
  ('internal medicine associates', '4450 31st ave s ste 102', '58104', 'tier 2'),
  ('eisenhower medical center', '39000 bob hope dr', '92270', 'tier 2'),
  ('shriners hospitals for children-salt lake city', '1275 fairfax rd', '84103', 'tier 2'),
  ('john muir medical center walnut creek', '1601 ygnacio valley rd', '94598', 'tier 2'),
  ('kaiser midatlantic', '1500 forest glenn rd, silver spring', '20901', 'tier 2'),
  ('christus st vincent regional medical center', '455 saint michaels dr santa fe', '87505', 'tier 2'),
  ('lexington health inc', '811 w main st, lexington', '29072', 'tier 2'),
  ('banner health', '2017 e ruby ln, phoenix', '85024', 'tier 2'),
  ('southwestern medical center', '5602 sw lee blvd', '73505', 'tier 2'),
  ('aurora west allis medical center', '8901 w lincoln ave', '53227', 'tier 2'),
  ('wk bossier health center', '2400 hospital dr', '71111', 'tier 2'),
  ('mission workwell', '310 long shoals rd', '28704', 'tier 2'),
  ('faith regional health services', '2700 w norfolk ave', '68701', 'tier 3'),
  ('st. catherine''s center for children', '30 n main ave', '12203', 'tier 3'),
  ('princeton nassau pediatrics p.a.', '301 n harrison st, princeton shopping center', '08540', 'tier 3'),
  ('lawrence general hospital', '1 general st', '01841', 'tier 3'),
  ('city of hope medical foundation', '1500 duarte rd', '91010', 'tier 3'),
  ('carle foundation hospital', '602 w university ave', '61801', 'tier 3'),
  ('pomona valley hospital medical center', '1798 n garey ave', '91767', 'tier 3'),
  ('lehigh valley hospital', '1200 s cedar crest blvd', '18103', 'tier 3'),
  ('synaptoveda pa', '4402 vance jackson rd', '78230', 'tier 3'),
  ('comanche county hospital cardiology', '3401 w gore blvd', '73505', 'tier 3'),
  ('mountainview hospital', '3100 n tenaya way, las vegas', '89128', 'tier 3'),
  ('pediatrix medical group of florida, inc.', '1301 concord ter', '33323', 'tier 3'),
  ('hazard arh regional medical center', '100 medical center dr', '41701', 'tier 3'),
  ('mercy medical center', '345 saint paul st', '21202', 'tier 3'),
  ('dallas pediatric neurology associates texas child neurology', '4032 mcdermott dr ste 100', '75024', 'tier 3'),
  ('concord family practice', '10430 lovell center dr', '37922', 'tier 3'),
  ('texas oncology pa, dallas', '3410 worth st', '75246', 'tier 3'),
  ('florida cancer specialists', '2501 n orange ave', '32804', 'tier 3'),
  ('overlake medical clinics llc', '1135 116th ave ne, suite 110', '98004', 'tier 3'),
  ('north jersey primary care associates', '30 prospect ave fl 3', '07601', 'tier 3'),
  ('indiana clinic-neurology llc', '355 w 16th st', '46202', 'tier 3'),
  ('wellstar medical group', '805 sandy plains rd', '30066', 'tier 3'),
  ('md kids pediatrics', '12377 merit dr ste 300', '75251', 'tier 3'),
  ('redlands community hospital', '350 terracina blvd', '92373', 'tier 3'),
  ('ascend hospice', '4550 w 109th st ste 210', '66211', 'tier 3'),
  ('utah valley pediatrics llc', '1355 n university ave, 210', '84604', 'tier 3'),
  ('franciscan st. james hospital and health centers', '20201 crawford ave', '60461', 'tier 3'),
  ('anne arundel medical center', '2001 medical pkwy', '21401', 'tier 3'),
  ('abc pediatric clinic p.a.', '13711 wallisville rd', '77049', 'tier 3'),
  ('riverside health system', '1905 w court st', '60901', 'tier 3'),
  ('towne centre surgery center llc', '4599 towne centre road', '48604', 'tier 3'),
  ('mercy medical center cedar rapids iowa', '701 10th st se', '52403', 'tier 3'),
  ('tuscaloosa pediatrics', '4880 harkey ln', '35406', 'tier 3'),
  ('perlman clinic', '9850 genesee ave ste 320', '92037', 'tier 3'),
  ('shults pediatrics p. c.', '9142 s northshore dr', '37922', 'tier 3'),
  ('tennessee cancer specialists pllc', '7650 dannaher dr', '37849', 'tier 3')
AS reporting_parent (
  reporting_parent_name,
  reporting_parent_address,
  reporting_parent_zi,
  reporting_parent_tier
);

In [0]:
select *
from reporting_parent limit 5

### Similarity Mapping

In [0]:
%python
!pip install rapidfuzz

In [0]:
%python
# ─────────────────────────────────────────────────────────────────────────────
# HCP → Reporting Parent Matching
# Matches pharmacist_hcps.hco_name/address against reporting_parent table
# using fuzzy similarity. No ZIP filtering — full cross-match with optimizations.
# ─────────────────────────────────────────────────────────────────────────────

# %pip install rapidfuzz tqdm

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG  — tune these to tighten or loosen matching
# ─────────────────────────────────────────────────────────────────────────────
SIMILARITY_THRESHOLD = 85   # Minimum weighted score to confirm a match (0–100)
NAME_WEIGHT          = 0.65 # Name carries slightly more weight
ADDRESS_WEIGHT       = 0.35

# ─────────────────────────────────────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────────────────────────────────────
import re
import pandas as pd
from rapidfuzz import fuzz, process
from tqdm import tqdm

# ─────────────────────────────────────────────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────────────────────────────────────────────
pharmacist_df     = spark.table("pharmacist_hcps").toPandas()
reporting_parent  = spark.table("reporting_parent").toPandas()

print(f"pharmacist_hcps   : {len(pharmacist_df):,} rows")
print(f"reporting_parent  : {len(reporting_parent):,} rows")

# ─────────────────────────────────────────────────────────────────────────────
# DATA CLEANING HELPERS  (US healthcare affiliations/claims specific)
# ─────────────────────────────────────────────────────────────────────────────

# Common US healthcare name abbreviations to expand for better matching
ABBREV_MAP = {
    r"\bmed\b"       : "medical",
    r"\bhosp\b"      : "hospital",
    r"\bhlth\b"      : "health",
    r"\bhealthcr\b"  : "healthcare",
    r"\bsys\b"       : "system",
    r"\bsvc\b"       : "services",
    r"\bsvcs\b"      : "services",
    r"\bctr\b"       : "center",
    r"\bctrs\b"      : "centers",
    r"\bdept\b"      : "department",
    r"\buniv\b"      : "university",
    r"\bregion\b"    : "regional",
    r"\brgn\b"       : "regional",
    r"\brehab\b"     : "rehabilitation",
    r"\bpeds\b"      : "pediatrics",
    r"\bped\b"       : "pediatric",
    r"\bphys\b"      : "physicians",
    r"\bpharm\b"     : "pharmacy",
    r"\bgen\b"       : "general",
    r"\bcomm\b"      : "community",
    r"\bpresb\b"     : "presbyterian",
    r"\bmem\b"       : "memorial",
    r"\bnorth\b"     : "north",
    r"\bsouth\b"     : "south",
    r"\beast\b"      : "east",
    r"\bwest\b"      : "west",
}

# Common US street suffix abbreviations for address normalization
STREET_ABBREV = {
    r"\bst\b"   : "street",
    r"\bave\b"  : "avenue",
    r"\bav\b"   : "avenue",
    r"\bblvd\b" : "boulevard",
    r"\bdr\b"   : "drive",
    r"\brd\b"   : "road",
    r"\bln\b"   : "lane",
    r"\bct\b"   : "court",
    r"\bpl\b"   : "place",
    r"\bpkwy\b" : "parkway",
    r"\bhwy\b"  : "highway",
    r"\bfwy\b"  : "freeway",
    r"\bsq\b"   : "square",
    r"\bste\b"  : "suite",
    r"\bflr\b"  : "floor",
}

def clean_name(text) -> str:
    """
    Normalize a healthcare org name:
      - lowercase, strip
      - remove punctuation except spaces
      - expand common abbreviations
      - collapse whitespace
    """
    if pd.isna(text) or str(text).strip() == "":
        return ""
    t = str(text).lower().strip()
    t = re.sub(r"[^\w\s]", " ", t)          # remove punctuation
    t = re.sub(r"\bllc\b|\binc\b|\bcorp\b|\bco\b|\bltd\b", "", t)  # strip legal suffixes
    for pattern, replacement in ABBREV_MAP.items():
        t = re.sub(pattern, replacement, t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

def clean_address(text) -> str:
    """
    Normalize a US street address:
      - lowercase, strip
      - remove punctuation
      - expand street type abbreviations
      - remove unit/suite numbers (noise for matching)
      - collapse whitespace
    """
    if pd.isna(text) or str(text).strip() == "":
        return ""
    t = str(text).lower().strip()
    t = re.sub(r"[^\w\s]", " ", t)
    t = re.sub(r"\b(suite|ste|unit|apt|floor|flr|bldg|building|#)\s*\w+", "", t)  # remove unit noise
    for pattern, replacement in STREET_ABBREV.items():
        t = re.sub(pattern, replacement, t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

# ─────────────────────────────────────────────────────────────────────────────
# APPLY CLEANING
# ─────────────────────────────────────────────────────────────────────────────
print("\nCleaning data...")

pharmacist_df["hco_name_clean"]    = pharmacist_df["hco_name"].apply(clean_name)
pharmacist_df["hco_address_clean"] = pharmacist_df["hco_address"].apply(clean_address)

reporting_parent["rp_name_clean"]    = reporting_parent["reporting_parent_name"].apply(clean_name)
reporting_parent["rp_address_clean"] = reporting_parent["reporting_parent_address"].apply(clean_address)

# ─────────────────────────────────────────────────────────────────────────────
# DEDUPLICATE PHARMACIST HCOs BEFORE MATCHING
# The pharmacist table has 124k rows but likely far fewer unique HCOs.
# Match at HCO level first, then join back to all HCP rows — much faster.
# ─────────────────────────────────────────────────────────────────────────────
print("Deduplicating unique HCOs from pharmacist table...")

unique_hcos = (
    pharmacist_df[["hco_name", "hco_address", "hco_name_clean", "hco_address_clean"]]
    .drop_duplicates(subset=["hco_name_clean", "hco_address_clean"])
    .reset_index(drop=True)
)

print(f"  Unique HCOs to match : {len(unique_hcos):,}  (down from {len(pharmacist_df):,} rows)")

# ─────────────────────────────────────────────────────────────────────────────
# PRE-COMPUTE REPORTING PARENT COMBINED KEY FOR RAPID CANDIDATE RETRIEVAL
# rapidfuzz.process.extractOne does a fast pre-filter on name alone,
# then we re-score with the weighted name+address formula.
# ─────────────────────────────────────────────────────────────────────────────
rp_names    = reporting_parent["rp_name_clean"].tolist()
rp_records  = reporting_parent.to_dict("records")   # list of dicts for O(1) access

# ─────────────────────────────────────────────────────────────────────────────
# MATCHING FUNCTION
# ─────────────────────────────────────────────────────────────────────────────
def match_hco(hco_name_clean: str, hco_address_clean: str):
    """
    1. Use rapidfuzz.process to get top-5 name candidates (fast C-level pre-filter).
    2. Re-score each candidate with weighted name + address similarity.
    3. Return best if score >= threshold, else None.
    """
    if not hco_name_clean:
        return None

    # Step 1: fast top-5 name candidates
    top_candidates = process.extract(
        hco_name_clean,
        rp_names,
        scorer=fuzz.token_set_ratio,
        limit=5,                        # check top 5 name matches
    )

    best_score  = -1.0
    best_idx    = None

    for _, name_score, idx in top_candidates:
        address_score = fuzz.token_set_ratio(hco_address_clean, rp_records[idx]["rp_address_clean"])
        combined      = NAME_WEIGHT * name_score + ADDRESS_WEIGHT * address_score
        if combined > best_score:
            best_score = combined
            best_idx   = idx

    if best_score >= SIMILARITY_THRESHOLD and best_idx is not None:
        rp = rp_records[best_idx]
        return {
            "matched_rp_name"    : rp["reporting_parent_name"],
            "matched_rp_address" : rp["reporting_parent_address"],
            "matched_rp_zip"     : rp["reporting_parent_zi"],
            "matched_rp_tier"    : rp["reporting_parent_tier"],
            "match_score"        : round(best_score, 2),
        }
    return None

# ─────────────────────────────────────────────────────────────────────────────
# RUN MATCHING ON UNIQUE HCOs  (with tqdm progress bar)
# ─────────────────────────────────────────────────────────────────────────────
print("\nRunning matching on unique HCOs...")

match_results = []

for _, row in tqdm(unique_hcos.iterrows(), total=len(unique_hcos), desc="Matching HCOs"):
    result = match_hco(row["hco_name_clean"], row["hco_address_clean"])
    if result:
        match_results.append({
            "hco_name"           : row["hco_name"],
            "hco_address"        : row["hco_address"],
            **result
        })

hco_match_df = pd.DataFrame(match_results)
print(f"\n  Unique HCOs matched : {len(hco_match_df):,} out of {len(unique_hcos):,}")

# ─────────────────────────────────────────────────────────────────────────────
# JOIN MATCHES BACK TO ALL HCP ROWS
# ─────────────────────────────────────────────────────────────────────────────
print("Joining matches back to full HCP table...")

output_df = pharmacist_df.merge(
    hco_match_df,
    on=["hco_name", "hco_address"],
    how="inner"                          # only keep HCPs with a confirmed match
).drop(columns=["hco_name_clean", "hco_address_clean"])

# Sort by match score for easy review
output_df = output_df.sort_values("match_score", ascending=False).reset_index(drop=True)

# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'─'*50}")
print(f"  Total HCPs in pharmacist table  : {len(pharmacist_df):,}")
print(f"  HCPs matched to a reporting parent : {len(output_df):,}")
print(f"  Unique reporting parents matched    : {output_df['matched_rp_name'].nunique():,}")
print(f"{'─'*50}\n")

display(output_df)

In [0]:
SELECT *
FROM pharmacist_hcps
WHERE LOWER(CONCAT(first_name__v, ' ', last_name__v)) IN (
    'neil patel',
    'richard dyke',
    'karen shalaby'
);

In [0]:
SELECT *
FROM pharmacist_hcps
WHERE LOWER(hco_name) LIKE '%children%philadelphia%'
  AND LOWER(CONCAT(first_name__v, ' ', last_name__v)) IN (
      'neil patel',
      'richard dyke',
      'karen shalaby'
);

In [0]:
SELECT *
FROM com_edp_prd.cmpa_insights_internal_schema.reference_file
WHERE LOWER(CONCAT(hcp_first_name, ' ', hcp_last_name)) IN (
    'neil patel',
    'richard dyke',
    'karen shalaby'
);

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.reference_file

In [0]:
select * from com_edp_prd.com_raw.vod_hcp
where first_name_cda__v ilike '%neil%' and last_name_cda__v ilike '%patel%' and vid__v in ('243053469240394756', '941662202607438687')